# R515B Soğutucu Akışkan — Model Eğitimi

Bu notebook, Doymuş ve Kızgın buhar veri setleri için **4 farklı senaryo** ile regresyon modeli eğitimi yapar:

| # | Senaryo | Açıklama |
|---|---------|----------|
| 1 | **Base** | Orijinal veri ile temel eğitim |
| 2 | **SMOGN** | SMOGN ile veri artırma sonrası eğitim |
| 3 | **STGP-EF** | SymbolicTransformer ve Evolutionary Forest Hibrit Yöntemi ile öznitelik inşası sonrası eğitim |
| 4 | **SMOGN+STGP-EF** | Önce SMOGN, sonra STGP-EF ile birleşik eğitim |

Her senaryo **11 farklı regresyon algoritması** ile değerlendirilir: RF, ET, AdaBoost, GBDT, DART, XGBoost, LightGBM, CatBoost, KNN, GP, EF.

---

## 1. Kütüphaneler ve Fonksiyonlar

In [1]:
import sys
sys.path.insert(0, 'R515B_Ozellikler')

import pandas as pd
from Functions import (
    run_saturated_analysis,
    run_superheated_analysis,
    build_results_table,
    show_best_results,
    compare_scenarios,
    target_summary,
    save_wide_results
)

c:\Users\SGUZEY\Documents\Yüksek Lisans Dersleri\Veri Mühendisliği\Data_Engineering_Project\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-05-19 20:38:50,346 - INFO - Loading faiss with AVX2 support.
2026-05-19 20:38:51,020 - INFO - Successfully loaded faiss with AVX2 support.


## 2. Veri Yükleme

In [2]:
df_doymus = pd.read_csv('R515B_Ozellikler/R515B_Doymus_Ozellikleri.csv')
df_kizgin = pd.read_csv('R515B_Ozellikler/R515B_Kizgin_Buhar.csv')

print(f'Doymuş Buhar: {df_doymus.shape}')
print(f'Kızgın Buhar: {df_kizgin.shape}')
print(f'\nDoymuş Sütunlar: {list(df_doymus.columns)}')
print(f'Kızgın Sütunlar: {list(df_kizgin.columns)}')

Doymuş Buhar: (136, 8)
Kızgın Buhar: (578, 5)

Doymuş Sütunlar: ['T(girdi)', 'P(çıktı)', 'v sıvı (çıktı)', 'v buhar (çıktı)', 'h sıvı (çıktı)', 'h buhar (çıktı)', 's sıvı (çıktı)', 's buhar (çıktı)']
Kızgın Sütunlar: ['T (girdi)', 'P (girdi)', 'v (çıktı)', 'h (çıktı)', 's (çıktı)']


In [3]:
df_doymus.head()

,T(girdi),P(çıktı),v sıvı (çıktı),v buhar (çıktı),h sıvı (çıktı),h buhar (çıktı),s sıvı (çıktı),s buhar (çıktı)
0,-30,61.2210,0.0007,0.2726,2.7282,198.2000,0.0446,0.8486
1,-29,64.2350,0.0007,0.2606,3.9422,198.8900,0.0496,0.8481
2,-28,67.3670,0.0007,0.2492,5.1587,199.5800,0.0546,0.8476
3,-27,70.6200,0.0007,0.2385,6.3775,200.2700,0.0595,0.8472
4,-26,73.9960,0.0008,0.2282,7.5988,200.9500,0.0645,0.8468


In [4]:
df_kizgin.head()

,T (girdi),P (girdi),v (çıktı),h (çıktı),s (çıktı)
0,-30,50,0.3357,198.5500,0.8639
1,-25,50,0.3433,202.4500,0.8798
2,-20,50,0.3508,206.4000,0.8956
3,-15,50,0.3582,210.4000,0.9112
4,-10,50,0.3656,214.4300,0.9267


---
## 3. Doymuş Buhar — Eğitim

**Girdi:** T(girdi)  
**Çıktılar (her biri ayrı ayrı):** P, v sıvı, v buhar, h sıvı, h buhar, s sıvı, s buhar

In [5]:
doymus_results = run_saturated_analysis(df_doymus)


▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  DOYMUŞ BUHAR ANALİZİ
▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓

  Target: P(çıktı)  |  Input: ['T(girdi)']
  [1/4] Base training...
  [2/4] SMOGN training...
Özel SMOGN Algoritması başlatılıyor (Custom Build)...
Özel SMOGN Başarılı! Orijinal: 108 -> Artırılmış/Dengelenmiş: 128


2026-05-19 20:41:22,834 - INFO - Hibrit Özellik İnşası (STGP-EF) Başlatılıyor...


  [3/4] STGP-EF training...


2026-05-19 20:41:30,397 - INFO - 
──────────────────────────────────────────────────────────────────────────────
2026-05-19 20:41:30,397 - INFO - SYMBOLIC TRANSFORMER (STGP) - Generated Features
2026-05-19 20:41:30,398 - INFO - ──────────────────────────────────────────────────────────────────────────────
2026-05-19 20:41:30,400 - INFO -   STGP_00: (((x0 * 0.352) * (0.007 - x0)) - x0)
2026-05-19 20:41:30,400 - INFO -   STGP_01: (((x0 * 0.352) * (0.007 - x0)) - x0)
2026-05-19 20:41:30,401 - INFO -   STGP_02: (((x0 * 0.352) * (0.007 - x0)) - x0)
2026-05-19 20:41:30,402 - INFO -   STGP_03: (((x0 * 0.352) * (0.007 - x0)) - x0)
2026-05-19 20:41:30,402 - INFO -   STGP_04: (((x0 * 0.352) * (0.007 - x0)) - x0)
2026-05-19 20:41:30,403 - INFO -   STGP_05: (((x0 * 0.352) * (0.007 - x0)) - x0)
2026-05-19 20:41:30,404 - INFO -   STGP_06: (((x0 * 0.352) * (0.007 - x0)) - x0)
2026-05-19 20:41:30,404 - INFO -   STGP_07: (((x0 * 0.352) * (0.007 - x0)) - x0)
2026-05-19 20:41:30,406 - INFO -   STGP_08: (

  [4/4] SMOGN + STGP-EF training...


...

  [3/4] STGP-EF training...


2026-05-19 21:39:53,344 - INFO - 
──────────────────────────────────────────────────────────────────────────────
2026-05-19 21:39:53,345 - INFO - SYMBOLIC TRANSFORMER (STGP) - Generated Features
2026-05-19 21:39:53,345 - INFO - ──────────────────────────────────────────────────────────────────────────────
2026-05-19 21:39:53,346 - INFO -   STGP_00: (((((x0 * x0) + (0.888 + (((x0 * x0) + (0.888 + (0.888 + (0.888 + x0)))) + x0))) + (0.888 + 0.678)) + (0.888 + x0)) * ((((x0 * -0.133) / (-0.233 / x0)) - (0.209 - -0.839)) * (0.888 + x0)))
2026-05-19 21:39:53,347 - INFO -   STGP_01: ((((0.888 + x0) + ((((x0 * x0) + (0.888 + (0.888 + x0))) + 0.888) + ((((x0 * x0) + (0.888 + x0)) + 0.888) + 0.888))) + 0.888) * ((((x0 * -0.133) / (-0.233 / x0)) - (0.209 - -0.839)) * (0.888 + x0)))
2026-05-19 21:39:53,347 - INFO -   STGP_02: ((((((x0 * -0.133) / (-0.233 / x0)) - (0.209 - -0.839)) + (0.888 + (0.888 + x0))) + 0.888) * ((((x0 * -0.133) / (-0.233 / x0)) - (0.209 - -0.839)) * (0.888 + x0)))
2026-05-1

  [4/4] SMOGN + STGP-EF training...


2026-05-19 21:42:49,674 - INFO - 
──────────────────────────────────────────────────────────────────────────────
2026-05-19 21:42:49,675 - INFO - SYMBOLIC TRANSFORMER (STGP) - Generated Features
2026-05-19 21:42:49,676 - INFO - ──────────────────────────────────────────────────────────────────────────────
2026-05-19 21:42:49,676 - INFO -   STGP_00: (((((x0 * -0.133) / (-0.233 / x0)) + (0.888 + x0)) + 0.888) * ((((x0 * -0.133) / (-0.233 / x0)) - (0.209 - -0.839)) * (0.888 + x0)))
2026-05-19 21:42:49,677 - INFO -   STGP_01: (((((x0 * -0.133) / (-0.233 / x0)) + 0.888) + (0.888 + x0)) * ((((x0 * -0.133) / (-0.233 / x0)) - (0.209 - -0.839)) * (0.888 + x0)))
2026-05-19 21:42:49,677 - INFO -   STGP_02: (((((x0 * x0) + 0.888) + 0.888) + (0.888 + x0)) * ((((x0 * -0.133) / (-0.233 / x0)) - (0.209 - -0.839)) * (0.888 + x0)))
2026-05-19 21:42:49,678 - INFO -   STGP_03: ((((((x0 * x0) + 0.888) + (0.888 + x0)) + 0.888) + (0.888 + x0)) * ((((x0 * -0.133) / (-0.233 / x0)) - (0.209 - -0.839)) * (0.888 

  Completed.



### 3.1 Doymuş Buhar — Sonuç Tablosu

In [6]:
df_doymus_results = build_results_table(doymus_results)
df_doymus_results

,Target,Scenario,Algorithm,Train_R2,Test_R2,Train_RMSE,Test_RMSE,Train_MAE,Test_MAE,Train_MAPE,Test_MAPE
0,P(çıktı),Base,AdaBoost,0.9988,0.9986,0.0343,0.0377,0.0242,0.0298,0.0509,0.0367
1,P(çıktı),Base,CatBoost,0.9998,0.9995,0.0129,0.0229,0.0101,0.0167,0.0337,0.0307
2,P(çıktı),Base,DART,0.9544,0.9610,0.2136,0.2011,0.1200,0.1150,0.1497,0.1097
3,P(çıktı),Base,EF,1.0000,1.0000,0.0000,0.0032,0.0000,0.0022,0.0000,0.0036
4,P(çıktı),Base,ET,1.0000,1.0000,0.0000,0.0025,0.0000,0.0017,0.0000,0.0024
...,...,...,...,...,...,...,...,...,...,...,...
303,s buhar (çıktı),SMOGN+STGP-EF,GP,0.9770,0.9139,0.1798,0.2183,0.1421,0.1533,0.5617,0.3698
304,s buhar (çıktı),SMOGN+STGP-EF,KNN,0.9953,0.9941,0.0817,0.0570,0.0287,0.0344,0.2245,0.1846
305,s buhar (çıktı),SMOGN+STGP-EF,LightGBM,0.9918,0.9691,0.1077,0.1309,0.0452,0.0792,0.1398,0.2515
306,s buhar (çıktı),SMOGN+STGP-EF,RF,0.9988,0.9964,0.0406,0.0446,0.0143,0.0229,0.0403,0.0582


### 3.2 Doymuş Buhar — Senaryo Karşılaştırması

In [7]:
compare_scenarios(df_doymus_results)

Scenario                   Base  SMOGN  STGP-EF  SMOGN+STGP-EF
Target         Algorithm                                      
P(çıktı)       AdaBoost  0.9986 0.9985   0.9982         0.9989
               CatBoost  0.9995 0.9992   0.9994         0.9995
               DART      0.9610 0.9897   0.9624         0.9908
               EF        1.0000 1.0000   1.0000         1.0000
               ET        1.0000 1.0000   1.0000         1.0000
...                         ...    ...      ...            ...
v sıvı (çıktı) GP        0.9541 0.9468   0.8351         0.8887
               KNN       0.9989 0.9985   0.9996         0.9996
               LightGBM  0.9429 0.9881   0.9448         0.9884
               RF        0.9992 0.9995   0.9999         0.9997
               XGBoost   0.9981 0.9984   0.9980         0.9984

[77 rows x 4 columns]

### 3.3 Doymuş Buhar — Hedef Bazlı Detay

Her bir hedef değişken için detaylı pivot tablo;

In [8]:
target_summary(df_doymus_results, 'P(çıktı)')

Scenario,Base,SMOGN,STGP-EF,SMOGN+STGP-EF
Algorithm,,,,
AdaBoost,0.9986,0.9985,0.9982,0.9989
CatBoost,0.9995,0.9992,0.9994,0.9995
DART,0.9610,0.9897,0.9624,0.9908
EF,1.0000,1.0000,1.0000,1.0000
ET,1.0000,1.0000,1.0000,1.0000
GBDT,0.9992,0.9993,0.9997,0.9995
GP,0.9737,0.9737,0.9903,0.9395
KNN,0.9997,0.9993,0.9998,0.9997
LightGBM,0.9710,0.9962,0.9938,0.9966


In [9]:
target_summary(df_doymus_results, 'v sıvı (çıktı)')

Scenario,Base,SMOGN,STGP-EF,SMOGN+STGP-EF
Algorithm,,,,
AdaBoost,0.9971,0.9947,0.9989,0.9971
CatBoost,0.9986,0.9992,0.9988,0.9994
DART,0.9345,0.9848,0.9343,0.9859
EF,1.0000,1.0000,1.0000,1.0000
ET,1.0000,1.0000,1.0000,1.0000
GBDT,0.9978,0.9990,0.9994,0.9995
GP,0.9541,0.9468,0.8351,0.8887
KNN,0.9989,0.9985,0.9996,0.9996
LightGBM,0.9429,0.9881,0.9448,0.9884


In [10]:
target_summary(df_doymus_results, 'v buhar (çıktı)')

Scenario,Base,SMOGN,STGP-EF,SMOGN+STGP-EF
Algorithm,,,,
AdaBoost,0.9909,0.9954,0.9922,0.9933
CatBoost,0.9971,0.9991,0.9990,0.9981
DART,0.8327,0.9670,0.8721,0.9661
EF,1.0000,1.0000,0.9998,1.0000
ET,1.0000,1.0000,1.0000,1.0000
GBDT,0.9965,0.9983,0.9986,0.9989
GP,0.9398,0.7151,0.8594,0.9785
KNN,0.9984,0.9993,0.9987,0.9993
LightGBM,0.8011,0.9530,0.8503,0.9481


In [11]:
target_summary(df_doymus_results, 'h sıvı (çıktı)')

Scenario,Base,SMOGN,STGP-EF,SMOGN+STGP-EF
Algorithm,,,,
AdaBoost,0.9993,0.9992,0.9997,0.9993
CatBoost,0.9996,0.9992,0.9996,0.9996
DART,0.9749,0.9901,0.9760,0.9918
EF,1.0000,1.0000,1.0000,1.0000
ET,1.0000,1.0000,1.0000,1.0000
GBDT,0.9993,0.9993,0.9996,0.9994
GP,0.9955,0.9955,0.9955,0.9955
KNN,0.9997,0.9994,0.9998,0.9995
LightGBM,0.9793,0.9964,0.9827,0.9987


In [12]:
target_summary(df_doymus_results, 'h buhar (çıktı)')

Scenario,Base,SMOGN,STGP-EF,SMOGN+STGP-EF
Algorithm,,,,
AdaBoost,0.9961,0.9953,0.9985,0.9984
CatBoost,0.9993,0.9991,0.9992,0.9993
DART,0.9765,0.9895,0.9839,0.9903
EF,1.0000,1.0000,0.9999,1.0000
ET,1.0000,1.0000,1.0000,1.0000
GBDT,0.9990,0.9990,0.9995,0.9994
GP,0.9518,0.9518,0.9518,0.9688
KNN,0.9994,0.9993,0.9995,0.9991
LightGBM,0.9747,0.9944,0.9911,0.9962


In [13]:
target_summary(df_doymus_results, 's sıvı (çıktı)')

Scenario,Base,SMOGN,STGP-EF,SMOGN+STGP-EF
Algorithm,,,,
AdaBoost,0.9996,0.9948,0.9991,0.9993
CatBoost,0.9995,0.9992,0.9996,0.9994
DART,0.9753,0.9901,0.9782,0.9919
EF,1.0000,1.0000,1.0000,1.0000
ET,1.0000,1.0000,1.0000,1.0000
GBDT,0.9993,0.9993,0.9995,0.9994
GP,0.9998,0.9998,0.9998,0.9998
KNN,0.9997,0.9994,0.9997,0.9996
LightGBM,0.9778,0.9960,0.9827,0.9968


In [14]:
target_summary(df_doymus_results, 's buhar (çıktı)')

Scenario,Base,SMOGN,STGP-EF,SMOGN+STGP-EF
Algorithm,,,,
AdaBoost,0.9163,0.8474,0.9716,0.9847
CatBoost,0.9793,0.9929,0.9904,0.9978
DART,0.4471,0.6337,0.8992,0.9477
EF,0.9999,0.9977,0.9990,0.9974
ET,0.9996,0.9975,0.9995,0.9972
GBDT,0.9737,0.9889,0.9966,0.9940
GP,0.0665,0.8213,0.9646,0.9139
KNN,0.9854,0.9929,0.9913,0.9941
LightGBM,0.4591,0.5960,0.9530,0.9691


---
## 4. Kızgın Buhar — Eğitim

**Girdiler:** T (girdi), P (girdi)  
**Çıktılar (her biri ayrı ayrı):** v, h, s

In [15]:
kizgin_results = run_superheated_analysis(df_kizgin)


▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  KIZGIN BUHAR ANALİZİ
▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓

  Target: v (çıktı)  |  Input: ['T (girdi)', 'P (girdi)']
  [1/4] Base training...
  [2/4] SMOGN training...
Özel SMOGN Algoritması başlatılıyor (Custom Build)...
Özel SMOGN Başarılı! Orijinal: 462 -> Artırılmış/Dengelenmiş: 538


2026-05-19 21:48:55,613 - INFO - Hibrit Özellik İnşası (STGP-EF) Başlatılıyor...


  [3/4] STGP-EF training...


2026-05-19 21:49:08,079 - INFO - 
──────────────────────────────────────────────────────────────────────────────
2026-05-19 21:49:08,080 - INFO - SYMBOLIC TRANSFORMER (STGP) - Generated Features
2026-05-19 21:49:08,082 - INFO - ──────────────────────────────────────────────────────────────────────────────
2026-05-19 21:49:08,082 - INFO -   STGP_00: (((x1 * x1) - (x1 + x1)) * ((((x1 * x1) - (x1 + x1)) * ((((x1 * x1) - (x1 + x1)) * (((x1 * x1) - (x1 + x1)) * ((((x1 * x1) - (x1 + x1)) * ((x1 * x1) - (x1 + x1))) - ((x1 * x1) - x0)))) - ((x1 * x1) - x0))) * (x1 * x1)))
2026-05-19 21:49:08,083 - INFO -   STGP_01: (((x1 * x1) - ((x1 + x1) + x1)) * ((x1 + (((x1 * x1) - (x1 + x1)) * (((x1 * x1) - (x1 + x1)) * ((((x1 * x1) - (x1 + x1)) * ((x1 * x1) - (x1 + x1))) - ((x1 * x1) - x0))))) * (x1 * x1)))
2026-05-19 21:49:08,084 - INFO -   STGP_02: (((x1 * x1) - (x1 + (x1 + x1))) * ((x1 + (((((x1 * x1) - (x1 + x1)) * ((x1 * x1) - (x1 + x1))) - (x1 + x1)) * ((((x1 * x1) - x1) * ((x1 * x1) - (x1 + x1))) 

  [4/4] SMOGN + STGP-EF training...


...

  [3/4] STGP-EF training...


2026-05-19 22:14:41,090 - INFO - 
──────────────────────────────────────────────────────────────────────────────
2026-05-19 22:14:41,091 - INFO - SYMBOLIC TRANSFORMER (STGP) - Generated Features
2026-05-19 22:14:41,092 - INFO - ──────────────────────────────────────────────────────────────────────────────
2026-05-19 22:14:41,092 - INFO -   STGP_00: (((x1 * (x0 * 0.31)) + x0) - x1)
2026-05-19 22:14:41,093 - INFO -   STGP_01: ((((x0 * 0.31) * x1) + x0) - x1)
2026-05-19 22:14:41,093 - INFO -   STGP_02: (((x1 * (x0 * 0.31)) + x0) - x1)
2026-05-19 22:14:41,094 - INFO -   STGP_03: (((x1 * (x0 * 0.31)) + x0) - x1)
2026-05-19 22:14:41,094 - INFO -   STGP_04: (x1 - ((x1 * (x0 * 0.31)) + x0))
2026-05-19 22:14:41,095 - INFO -   STGP_05: (((x1 * (x0 * 0.31)) + x0) - x1)
2026-05-19 22:14:41,096 - INFO -   STGP_06: ((((x0 * 0.31) * x1) + x0) - x1)
2026-05-19 22:14:41,097 - INFO -   STGP_07: (((x1 * (x0 * 0.31)) + x0) - x1)
2026-05-19 22:14:41,097 - INFO -   STGP_08: (((x1 * (x0 * 0.31)) + x0) - x1)


  [4/4] SMOGN + STGP-EF training...


2026-05-19 22:18:42,905 - INFO - 
──────────────────────────────────────────────────────────────────────────────
2026-05-19 22:18:42,906 - INFO - SYMBOLIC TRANSFORMER (STGP) - Generated Features
2026-05-19 22:18:42,906 - INFO - ──────────────────────────────────────────────────────────────────────────────
2026-05-19 22:18:42,906 - INFO -   STGP_00: (((x1 * (x0 * 0.31)) + x0) - x1)
2026-05-19 22:18:42,907 - INFO -   STGP_01: (((x1 * (x0 * 0.31)) + x0) - x1)
2026-05-19 22:18:42,908 - INFO -   STGP_02: (((x1 * (x0 * 0.31)) + x0) - x1)
2026-05-19 22:18:42,909 - INFO -   STGP_03: (((x1 * (x0 * 0.31)) + x0) - x1)
2026-05-19 22:18:42,909 - INFO -   STGP_04: (((x1 * (x0 * 0.31)) + x0) - x1)
2026-05-19 22:18:42,910 - INFO -   STGP_05: (((x1 * (x0 * 0.31)) + x0) - x1)
2026-05-19 22:18:42,910 - INFO -   STGP_06: (((x1 * (x0 * 0.31)) + x0) - x1)
2026-05-19 22:18:42,911 - INFO -   STGP_07: (((x1 * (x0 * 0.31)) + x0) - x1)
2026-05-19 22:18:42,911 - INFO -   STGP_08: (((x1 * (x0 * 0.31)) + x0) - x1)


  Completed.



### 4.1 Kızgın Buhar — Sonuç Tablosu

In [16]:
df_kizgin_results = build_results_table(kizgin_results)
df_kizgin_results

,Target,Scenario,Algorithm,Train_R2,Test_R2,Train_RMSE,Test_RMSE,Train_MAE,Test_MAE,Train_MAPE,Test_MAPE
0,v (çıktı),Base,AdaBoost,0.9941,0.9822,0.0770,0.1095,0.0633,0.0829,0.3407,0.2540
1,v (çıktı),Base,CatBoost,0.9999,0.9979,0.0089,0.0379,0.0060,0.0148,0.0331,0.0372
2,v (çıktı),Base,DART,0.9820,0.9908,0.1342,0.0786,0.0632,0.0516,0.1270,0.1050
3,v (çıktı),Base,EF,1.0000,0.9998,0.0004,0.0116,0.0002,0.0047,0.0007,0.0150
4,v (çıktı),Base,ET,1.0000,0.9998,0.0000,0.0129,0.0000,0.0046,0.0000,0.0140
...,...,...,...,...,...,...,...,...,...,...,...
127,s (çıktı),SMOGN+STGP-EF,GP,0.8123,0.8682,0.5222,0.3183,0.2966,0.1856,0.3603,0.7581
128,s (çıktı),SMOGN+STGP-EF,KNN,0.9938,0.9816,0.0949,0.1188,0.0502,0.0567,0.1001,0.2475
129,s (çıktı),SMOGN+STGP-EF,LightGBM,0.9997,0.9916,0.0192,0.0805,0.0108,0.0446,0.0382,0.4733
130,s (çıktı),SMOGN+STGP-EF,RF,0.9994,0.9831,0.0283,0.1138,0.0144,0.0529,0.0645,0.7194


### 4.2 Kızgın Buhar — Senaryo Karşılaştırması

In [17]:
compare_scenarios(df_kizgin_results)

Scenario              Base  SMOGN  STGP-EF  SMOGN+STGP-EF
Target    Algorithm                                      
h (çıktı) AdaBoost  0.9703 0.9685   0.9978         0.9959
          CatBoost  0.9988 0.9987   0.9993         0.9992
          DART      0.9776 0.9833   0.9856         0.9916
          EF        0.9998 0.9998   0.9998         0.9998
          ET        0.9997 0.9997   0.9997         0.9997
          GBDT      0.9981 0.9982   0.9994         0.9995
          GP        0.9890 0.9882   0.9867         0.9882
          KNN       0.9973 0.9956   0.9994         0.9991
          LightGBM  0.9859 0.9912   0.9941         0.9963
          RF        0.9985 0.9982   0.9995         0.9996
          XGBoost   0.9988 0.9979   0.9989         0.9991
s (çıktı) AdaBoost  0.8655 0.8695   0.9861         0.9660
          CatBoost  0.9959 0.9951   0.9976         0.9973
          DART      0.9595 0.9529   0.9879         0.9798
          EF        0.9979 0.9967   0.9958         0.9953
          ET        0.9966 0.9969   0.9955         0.9944
          GBDT      0.9911 0.9931   0.9955         0.9958
          GP        0.7208 0.7635   0.8682         0.8682
          KNN       0.9686 0.9565   0.9884         0.9816
          LightGBM  0.9771 0.9660   0.9952         0.9916
          RF        0.9846 0.9890   0.9943         0.9831
          XGBoost   0.9941 0.9900   0.9935         0.9847
v (çıktı) AdaBoost  0.9822 0.9741   0.9923         0.9866
          CatBoost  0.9979 0.9990   0.9993         0.9987
          DART      0.9908 0.9836   0.9952         0.9915
          EF        0.9998 0.9985   0.9999         0.9992
          ET        0.9998 0.9990   0.9999         0.9994
          GBDT      0.9993 0.9988   0.9994         0.9989
          GP        0.0762 0.3710  -0.1859         0.6884
          KNN       0.7521 0.6422   0.9988         0.9993
          LightGBM  0.9961 0.9963   0.9975         0.9983
          RF        0.9991 0.9974   0.9996         0.9991
          XGBoost   0.9981 0.9975   0.9978         0.9961

### 4.3 Doymuş Buhar — Hedef Bazlı Detay

Her bir hedef değişken için detaylı pivot tablo;

In [18]:
target_summary(df_kizgin_results, 'v (çıktı)')

Scenario,Base,SMOGN,STGP-EF,SMOGN+STGP-EF
Algorithm,,,,
AdaBoost,0.9822,0.9741,0.9923,0.9866
CatBoost,0.9979,0.9990,0.9993,0.9987
DART,0.9908,0.9836,0.9952,0.9915
EF,0.9998,0.9985,0.9999,0.9992
ET,0.9998,0.9990,0.9999,0.9994
GBDT,0.9993,0.9988,0.9994,0.9989
GP,0.0762,0.3710,-0.1859,0.6884
KNN,0.7521,0.6422,0.9988,0.9993
LightGBM,0.9961,0.9963,0.9975,0.9983


In [19]:
target_summary(df_kizgin_results, 'h (çıktı)')

Scenario,Base,SMOGN,STGP-EF,SMOGN+STGP-EF
Algorithm,,,,
AdaBoost,0.9703,0.9685,0.9978,0.9959
CatBoost,0.9988,0.9987,0.9993,0.9992
DART,0.9776,0.9833,0.9856,0.9916
EF,0.9998,0.9998,0.9998,0.9998
ET,0.9997,0.9997,0.9997,0.9997
GBDT,0.9981,0.9982,0.9994,0.9995
GP,0.9890,0.9882,0.9867,0.9882
KNN,0.9973,0.9956,0.9994,0.9991
LightGBM,0.9859,0.9912,0.9941,0.9963


In [20]:
target_summary(df_kizgin_results, 's (çıktı)')

Scenario,Base,SMOGN,STGP-EF,SMOGN+STGP-EF
Algorithm,,,,
AdaBoost,0.8655,0.8695,0.9861,0.9660
CatBoost,0.9959,0.9951,0.9976,0.9973
DART,0.9595,0.9529,0.9879,0.9798
EF,0.9979,0.9967,0.9958,0.9953
ET,0.9966,0.9969,0.9955,0.9944
GBDT,0.9911,0.9931,0.9955,0.9958
GP,0.7208,0.7635,0.8682,0.8682
KNN,0.9686,0.9565,0.9884,0.9816
LightGBM,0.9771,0.9660,0.9952,0.9916


# En İyi Sonuçlar

Her hedef değişken için en iyi (Algoritma, Senaryo) kombinasyonunu döndürür.

In [21]:
show_best_results(df_doymus_results)

,Target,Scenario,Algorithm,Test_R2,Test_RMSE,Test_MAE,Test_MAPE
0,P(çıktı),STGP-EF,ET,1.0000,0.0023,0.0018,0.0037
1,h buhar (çıktı),STGP-EF,ET,1.0000,0.0019,0.0014,0.0049
2,h sıvı (çıktı),STGP-EF,ET,1.0000,0.0019,0.0016,0.0135
3,s buhar (çıktı),Base,EF,0.9999,0.0061,0.0046,0.0210
4,s sıvı (çıktı),STGP-EF,ET,1.0000,0.0020,0.0016,0.0046
5,v buhar (çıktı),STGP-EF,ET,1.0000,0.0035,0.0021,0.0615
6,v sıvı (çıktı),Base,EF,1.0000,0.0032,0.0019,0.0046


---
## 5. Genel Karşılaştırma

In [22]:

df_doymus_results['Veri Seti'] = 'Doymuş Buhar'
df_kizgin_results['Veri Seti'] = 'Kızgın Buhar'
df_all = pd.concat([df_doymus_results, df_kizgin_results], ignore_index=True)

print(f'Toplam değerlendirme sayısı: {len(df_all)}')
df_all.head(20)

Toplam değerlendirme sayısı: 440


,Target,Scenario,Algorithm,Train_R2,Test_R2,Train_RMSE,Test_RMSE,Train_MAE,Test_MAE,Train_MAPE,Test_MAPE,Veri Seti
0,P(çıktı),Base,AdaBoost,0.9988,0.9986,0.0343,0.0377,0.0242,0.0298,0.0509,0.0367,Doymuş Buhar
1,P(çıktı),Base,CatBoost,0.9998,0.9995,0.0129,0.0229,0.0101,0.0167,0.0337,0.0307,Doymuş Buhar
2,P(çıktı),Base,DART,0.9544,0.9610,0.2136,0.2011,0.1200,0.1150,0.1497,0.1097,Doymuş Buhar
3,P(çıktı),Base,EF,1.0000,1.0000,0.0000,0.0032,0.0000,0.0022,0.0000,0.0036,Doymuş Buhar
4,P(çıktı),Base,ET,1.0000,1.0000,0.0000,0.0025,0.0000,0.0017,0.0000,0.0024,Doymuş Buhar
5,P(çıktı),Base,GBDT,1.0000,0.9992,0.0006,0.0292,0.0004,0.0219,0.0015,0.0368,Doymuş Buhar
6,P(çıktı),Base,GP,0.9728,0.9737,0.1650,0.1651,0.1223,0.1381,0.3673,0.2629,Doymuş Buhar
7,P(çıktı),Base,KNN,0.9995,0.9997,0.0226,0.0168,0.0100,0.0129,0.0212,0.0228,Doymuş Buhar
8,P(çıktı),Base,LightGBM,0.9617,0.9710,0.1958,0.1734,0.0938,0.0876,0.1242,0.0950,Doymuş Buhar
9,P(çıktı),Base,RF,0.9999,0.9997,0.0082,0.0168,0.0055,0.0118,0.0203,0.0197,Doymuş Buhar


In [24]:
# Senaryo bazlı ortalama Test R² skorları
df_all.groupby(['Veri Seti', 'Scenario'])['Test_R2'].mean().unstack()

Scenario,Base,SMOGN,SMOGN+STGP-EF,STGP-EF
Veri Seti,,,,
Doymuş Buhar,0.9592,0.9768,0.9917,0.9843
Kızgın Buhar,0.9441,0.9499,0.9806,0.9561


In [25]:
# Algoritma bazlı ortalama Test R² skorları
df_all.groupby(['Veri Seti', 'Algorithm'])['Test_R2'].mean().unstack()

Algorithm,AdaBoost,CatBoost,DART,EF,ET,GBDT,GP,KNN,LightGBM,RF,XGBoost
Veri Seti,,,,,,,,,,,
Doymuş Buhar,0.9876,0.9979,0.9328,0.9998,0.9998,0.9975,0.9131,0.9982,0.9364,0.9989,0.9961
Kızgın Buhar,0.9629,0.9981,0.9816,0.9985,0.9983,0.9972,0.6769,0.9399,0.9905,0.9952,0.9955


In [26]:
# Geniş formatlı sonuç tablosu oluştur ve kaydet
df_wide = save_wide_results(df_all, 'Results/Model_Training_Results.csv')
print(f'Format: Geniş tablo — {len(df_wide)} satır, {len(df_wide.columns)} sütun')
df_wide

Format: Geniş tablo — 110 satır, 37 sütun


,Veri Seti,Target,Algorithm,Base_Train_R2,Base_Test_R2,Base_Train_RMSE,Base_Test_RMSE,Base_Train_MAE,Base_Test_MAE,Base_Train_MAPE,Base_Test_MAPE,SMOGN_Train_R2,SMOGN_Test_R2,SMOGN_Train_RMSE,SMOGN_Test_RMSE,SMOGN_Train_MAE,SMOGN_Test_MAE,SMOGN_Train_MAPE,SMOGN_Test_MAPE,STGP-EF_Train_R2,STGP-EF_Test_R2,STGP-EF_Train_RMSE,STGP-EF_Test_RMSE,STGP-EF_Train_MAE,STGP-EF_Test_MAE,STGP-EF_Train_MAPE,STGP-EF_Test_MAPE,SMOGN+STGP-EF_Train_R2,SMOGN+STGP-EF_Test_R2,SMOGN+STGP-EF_Train_RMSE,SMOGN+STGP-EF_Test_RMSE,SMOGN+STGP-EF_Train_MAE,SMOGN+STGP-EF_Test_MAE,SMOGN+STGP-EF_Train_MAPE,SMOGN+STGP-EF_Test_MAPE,Max_Test_R2,Max_Scenario
0,Doymuş Buhar,P(çıktı),AdaBoost,0.9988,0.9986,0.0343,0.0377,0.0242,0.0298,0.0509,0.0367,0.9988,0.9985,0.0401,0.0391,0.0300,0.0273,0.0667,0.0371,0.9983,0.9982,0.0410,0.0433,0.0291,0.0362,0.0682,0.0473,0.9988,0.9989,0.0399,0.0335,0.0278,0.0269,0.0514,0.0407,0.9989,SMOGN+STGP-EF
1,Doymuş Buhar,P(çıktı),CatBoost,0.9998,0.9995,0.0129,0.0229,0.0101,0.0167,0.0337,0.0307,0.9999,0.9992,0.0135,0.0281,0.0102,0.0207,0.0347,0.0520,0.9998,0.9994,0.0130,0.0244,0.0094,0.0191,0.0600,0.0349,0.9999,0.9995,0.0131,0.0230,0.0094,0.0173,0.0307,0.0343,0.9995,Base
2,Doymuş Buhar,P(çıktı),DART,0.9544,0.9610,0.2136,0.2011,0.1200,0.1150,0.1497,0.1097,0.9843,0.9897,0.1469,0.1035,0.1038,0.0805,0.1380,0.0969,0.9568,0.9624,0.2078,0.1974,0.1152,0.1168,0.1302,0.1041,0.9846,0.9908,0.1453,0.0979,0.1022,0.0721,0.1197,0.0739,0.9908,SMOGN+STGP-EF
3,Doymuş Buhar,P(çıktı),EF,1.0000,1.0000,0.0000,0.0032,0.0000,0.0022,0.0000,0.0036,1.0000,1.0000,0.0000,0.0056,0.0000,0.0041,0.0000,0.0057,1.0000,1.0000,0.0000,0.0033,0.0000,0.0024,0.0000,0.0060,1.0000,1.0000,0.0000,0.0063,0.0000,0.0044,0.0000,0.0057,1.0000,Base
4,Doymuş Buhar,P(çıktı),ET,1.0000,1.0000,0.0000,0.0025,0.0000,0.0017,0.0000,0.0024,1.0000,1.0000,0.0000,0.0036,0.0000,0.0024,0.0000,0.0030,1.0000,1.0000,0.0000,0.0023,0.0000,0.0018,0.0000,0.0037,1.0000,1.0000,0.0000,0.0033,0.0000,0.0024,0.0000,0.0029,1.0000,STGP-EF
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105,Kızgın Buhar,v (çıktı),GP,0.0490,0.0762,0.9752,0.7890,0.4045,0.3528,1.1577,0.7253,0.3352,0.3710,1.0249,0.6510,0.5212,0.3669,2.0898,1.4398,-0.1560,-0.1859,1.0752,0.8939,0.5052,0.4585,2.1415,1.2309,0.6450,0.6884,0.7490,0.4582,0.3756,0.2286,0.5622,0.4985,0.6884,SMOGN+STGP-EF
106,Kızgın Buhar,v (çıktı),KNN,0.9134,0.7521,0.2943,0.4087,0.1050,0.1556,0.2754,0.2783,0.8952,0.6422,0.4070,0.4910,0.1778,0.2086,0.3188,0.4051,0.9996,0.9988,0.0193,0.0285,0.0072,0.0157,0.0316,0.0464,0.9970,0.9993,0.0685,0.0210,0.0191,0.0130,0.0325,0.0410,0.9993,SMOGN+STGP-EF
107,Kızgın Buhar,v (çıktı),LightGBM,0.9964,0.9961,0.0599,0.0511,0.0207,0.0216,0.0784,0.0572,0.9973,0.9963,0.0656,0.0500,0.0226,0.0195,0.0276,0.0352,0.9981,0.9975,0.0436,0.0413,0.0121,0.0172,0.0299,0.0366,0.9994,0.9983,0.0311,0.0338,0.0089,0.0148,0.0096,0.0372,0.9983,SMOGN+STGP-EF
108,Kızgın Buhar,v (çıktı),RF,1.0000,0.9991,0.0070,0.0242,0.0032,0.0114,0.0148,0.0329,0.9994,0.9974,0.0309,0.0417,0.0094,0.0190,0.0220,0.0612,1.0000,0.9996,0.0056,0.0172,0.0027,0.0095,0.0139,0.0298,0.9996,0.9991,0.0256,0.0251,0.0067,0.0121,0.0110,0.0357,0.9996,STGP-EF
